In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO

def finviz_screener_scraper(filters: dict, page: int = 1):
    """
    Realiza web scraping do Finviz sem depender de arquivo de settings.
    """
    base_url = "https://finviz.com/screener.ashx?"
    
    # Parâmetros base do Finviz: v=111 (visão geral), ft=4 (filtros avançados)
    params = {'v': '111', 'ft': '4', 'r': (page-1)*20+1} 
    
    # 1. Mapeamento de filtros (Embutido para facilitar)
    # Se quiser adicionar mais, siga o padrão: 'Nome Amigável': 'prefixo_no_finviz'
    FINVIZ_MAP = {
        "IPO Date": "ipodate_",
        "Country": "ctry_",
        "Sector": "sec_",
        "Industry": "ind_",
        "Index": "idx_"
    }

    filter_list = []
    for key, value in filters.items():
        # Se a chave estiver no mapa, usa o prefixo, senão usa a chave como está
        prefix = FINVIZ_MAP.get(key, key.lower().replace(' ', '') + "_")
        
        # Formata o código do filtro (ex: ipodate_more25)
        clean_value = str(value).lower().replace(' ', '').replace('-', '')
        filter_list.append(f"{prefix}{clean_value}")

    if filter_list:
        params['f'] = ",".join(filter_list)
    
    # 2. Construir URL e Fazer Request
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(base_url, params=params, headers=headers)
        response.raise_for_status() 
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # O Finviz coloca os resultados em um <tr> com id 'screener-table'
        screener_tr = soup.find('tr', id='screener-table')
        if not screener_tr:
            return pd.DataFrame()
            
        table_container = screener_tr.find('table', class_='styled-table-new')
        if not table_container:
            return pd.DataFrame()
        
        # 3. Processar a Tabela
        df_list = pd.read_html(StringIO(str(table_container)))
        if not df_list:
            return pd.DataFrame()
            
        result_df = df_list[0]
        
        # Renomear colunas para o padrão
        result_df.columns = ["No.", "Ticker", "Company", "Sector", "Industry", "Country", 
                             "Market Cap", "P/E", "Price", "Change", "Volume"]
        
        return result_df

    except Exception as e:
        print(f"Erro ao processar página {page}: {e}")
        return pd.DataFrame()

def finviz_screener_scraper_all_pages(filters: dict):
    page = 1
    all_results = []
    
    while True:
        print(f"Coletando página {page}...")
        df = finviz_screener_scraper(filters, page=page)
        
        if df.empty:
            break
            
        all_results.append(df)
        page += 1
        
        # O Finviz costuma limitar o número de resultados. 
        # Se a página vier com menos de 20 linhas, é a última.
        if len(df) < 20:
            break
            
    if not all_results:
        return pd.DataFrame()
        
    return pd.concat(all_results).reset_index(drop=True)

In [2]:
# A chave "IPO Date" vai virar "ipodate_" e o valor "more25" vai completar
filtros = {
    "IPO Date": "more25"
}

df_finance = finviz_screener_scraper_all_pages(filtros)
print(df_finance.head())

Coletando página 1...
Coletando página 2...
Coletando página 3...
Coletando página 4...
Coletando página 5...
Coletando página 6...
Coletando página 7...
Coletando página 8...
Coletando página 9...
Coletando página 10...
Coletando página 11...
Coletando página 12...
Coletando página 13...
Coletando página 14...
Coletando página 15...
Coletando página 16...
Coletando página 17...
Coletando página 18...
Coletando página 19...
Coletando página 20...
Coletando página 21...
Coletando página 22...
Coletando página 23...
Coletando página 24...
Coletando página 25...
Coletando página 26...
Coletando página 27...
Coletando página 28...
Coletando página 29...
Coletando página 30...
Coletando página 31...
Coletando página 32...
Coletando página 33...
Coletando página 34...
Coletando página 35...
Coletando página 36...
Coletando página 37...
Coletando página 38...
Coletando página 39...
Coletando página 40...
Coletando página 41...
Coletando página 42...
Coletando página 43...
Coletando página 44.

In [3]:
df_finance

,No.,Ticker,Company,Sector,Industry,Country,Market Cap,P/E,Price,Change,Volume
0,1,A,Agilent Technologies Inc,Healthcare,Diagnostics & Research,USA,32.36B,25.25,114.52,-0.89%,1272523
1,2,AAME,Atlantic American Corp,Financial,Insurance - Life,USA,52.62M,11.45,2.58,-1.15%,3328
2,3,AAON,AAON Inc,Industrials,Building Products & Equipment,USA,7.66B,72.30,93.59,0.30%,491232
3,4,AAPL,Apple Inc,Technology,Consumer Electronics,USA,4112.77B,33.89,280.14,3.24%,79913656
4,5,AB,AllianceBernstein Holding Lp,Financial,Asset Management,USA,3.65B,12.14,39.51,-0.98%,261854
...,...,...,...,...,...,...,...,...,...,...,...
1883,1884,YUM,Yum Brands Inc,Consumer Cyclical,Restaurants,USA,43.73B,25.52,158.36,-0.81%,1589091
1884,1885,ZBRA,Zebra Technologies Corp,Technology,Communication Equipment,USA,10.98B,27.81,227.08,0.36%,655914
1885,1886,ZD,Ziff Davis Inc,Communication Services,Advertising Agencies,USA,1.74B,41.66,46.10,0.74%,643614
1886,1887,ZION,Zions Bancorporation N.A,Financial,Banks - Regional,USA,9.30B,9.83,63.26,-0.25%,1340652


In [40]:
df_finance

,No.,Ticker,Company,Sector,Industry,Country,Market Cap,P/E,Price,Change,Volume
0,1,A,Agilent Technologies Inc,Healthcare,Diagnostics & Research,USA,32.36B,25.25,114.52,-0.89%,1272523
1,2,AAME,Atlantic American Corp,Financial,Insurance - Life,USA,52.62M,11.45,2.58,-1.15%,3328
2,3,AAON,AAON Inc,Industrials,Building Products & Equipment,USA,7.66B,72.30,93.59,0.30%,491232
3,4,AAPL,Apple Inc,Technology,Consumer Electronics,USA,4112.77B,33.89,280.14,3.24%,79913656
4,5,AB,AllianceBernstein Holding Lp,Financial,Asset Management,USA,3.65B,12.14,39.51,-0.98%,261854
...,...,...,...,...,...,...,...,...,...,...,...
1883,1884,YUM,Yum Brands Inc,Consumer Cyclical,Restaurants,USA,43.73B,25.52,158.36,-0.81%,1589091
1884,1885,ZBRA,Zebra Technologies Corp,Technology,Communication Equipment,USA,10.98B,27.81,227.08,0.36%,655914
1885,1886,ZD,Ziff Davis Inc,Communication Services,Advertising Agencies,USA,1.74B,41.66,46.10,0.74%,643614
1886,1887,ZION,Zions Bancorporation N.A,Financial,Banks - Regional,USA,9.30B,9.83,63.26,-0.25%,1340652


In [47]:
from numpy.random import random

df_finance = df_finance[df_finance["Country"]=="USA"]
df_finance = df_finance[df_finance["P/E"]!="-"]
df_finance = df_finance[df_finance["Market Cap"]!="-"]

In [68]:
import pandas as pd
import numpy as np

def robust_convert_mktcap(val):
    if pd.isna(val) or val == '-': return 0.0
    val = str(val).upper().strip()
    try:
        if 'T' in val: return float(val.replace('T', '')) * 1e12 # Trillion
        if 'B' in val: return float(val.replace('B', '')) * 1e9  # Billion
        if 'M' in val: return float(val.replace('M', '')) * 1e6  # Million
        return float(val)
    except:
        return 0.0

df_finance['Market Cap Value'] = df_finance['Market Cap'].apply(robust_convert_mktcap)

def get_balanced_sample(df, n_samples=50):
    # Garante que os valores são numéricos
    df['Market Cap Value'] = df['Market Cap'].apply(robust_convert_mktcap)
    
    # 1. Pega as 10 maiores (Hubs garantidos para o grafo)
    large_hubs = df.nlargest(10, 'Market Cap Value')
    
    # 2. Pega o restante de forma aleatória e diversificada por setor
    remaining = df.drop(large_hubs.index)
    
    # Amostra o restante para completar n_samples
    others = remaining.sample(n_samples - 10)
    
    final_sample = pd.concat([large_hubs, others]).drop_duplicates()
    return final_sample.reset_index(drop=True)

# Execute para testar
df_sample = get_balanced_sample(df_finance, 1000)

In [69]:
df_sample

,No.,Ticker,Company,Sector,Industry,Country,Market Cap,P/E,Price,Change,Volume,Market Cap Value
0,1236,NVDA,NVIDIA Corp,Technology,Semiconductors,USA,4822.33B,40.49,198.45,-0.56%,128641576,4.822330e+12
1,4,AAPL,Apple Inc,Technology,Consumer Electronics,USA,4112.77B,33.89,280.14,3.24%,79913656,4.112770e+12
2,1131,MSFT,Microsoft Corp,Technology,Software - Infrastructure,USA,3078.64B,24.68,414.44,1.63%,31372150,3.078640e+12
3,89,AMZN,Amazon.com Inc,Consumer Cyclical,Internet Retail,USA,2884.94B,32.06,268.26,1.21%,50823680,2.884940e+12
4,1838,WMT,Walmart Inc,Consumer Defensive,Discount Stores,USA,1049.17B,48.21,131.60,-0.25%,10480768,1.049170e+12
...,...,...,...,...,...,...,...,...,...,...,...,...
995,1666,TLF,Tandy Leather Factory Inc,Consumer Cyclical,Specialty Retail,USA,18.89M,2.00,2.34,0.00%,3752,1.889000e+07
996,1607,STZ,Constellation Brands Inc,Consumer Defensive,Beverages - Brewers,USA,26.31B,15.94,152.82,-2.40%,1503772,2.631000e+10
997,1664,TKO,TKO Group Holdings Inc,Communication Services,Entertainment,USA,36.10B,81.14,185.95,-0.08%,1028193,3.610000e+10
998,1051,MBWM,Mercantile Bank Corp,Financial,Banks - Regional,USA,897.60M,9.32,51.96,1.27%,133422,8.976000e+08


In [ ]:
df_sample[["Ticker", "Sector", "Industry", "Country"]].to_parquet("../../data/02_clean/metadata_att.parquet")
df_sample[["Ticker", "Sector", "Industry", "Country"]].to_csv("metadata_att.csv")

In [72]:
import yfinance as yf 

returns = yf.download(
    tickers=df_sample["Ticker"].to_list(),
    start='1986-01-01',
    end='2025-12-31',
    auto_adjust=True
)["Close"]

[*********************100%***********************]  1000 of 1000 completed

1 Failed download:
['AFL']: TypeError("'NoneType' object is not subscriptable")


In [73]:
returns = returns.pct_change()[1:]

C:\Users\madug\AppData\Local\Temp\ipykernel_23996\4017666420.py:1: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = returns.pct_change()[1:]


In [85]:
returns_filtered = returns.loc[:, ~returns.isna().any()]

In [88]:
df_sample_filtered = df_sample[df_sample["Ticker"].isin(returns_filtered.columns)]

In [89]:
df_sample_filtered[["Ticker", "Sector", "Industry", "Country"]].to_parquet("../../data/02_clean/metadata_att.parquet")
df_sample_filtered[["Ticker", "Sector", "Industry", "Country"]].to_csv("metadata_att.csv")

In [90]:
returns_filtered.to_parquet("../../data/02_clean/returns_30_years.parquet")

## Salvando por ano

In [11]:
import pandas as pd 
import numpy as np
from datetime import date

returns = pd.read_parquet("../../data/02_clean/returns_30_years.parquet")

# Dividing by years
for year in range(2015,2026):
    for k in [1, 10, 30]:
        returns_year = returns.loc[f"{year-k+1}-01-01":f"{year}-12-31"]
        returns_year.to_parquet(f"../../data/02_clean/returns_new_{year-k+1}_{year}.parquet")

In [16]:
import pandas as pd 
import numpy as np
from datetime import date

returns = pd.read_parquet("../../data/02_clean/returns_30_years.parquet")

# Dividing by years
for year in range(2015,2026):
    returns_year = returns.loc[f"{year}-01-01":f"{year}-12-31"]
    returns_year.to_parquet(f"../../data/02_clean/returns_new_{year}.parquet")